In [ ]:
import os
from datasets import load_dataset

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))

HF_HOME: /mnt/New Volume/hf_home
HF_DATASETS_CACHE: /mnt/New Volume/hf_home/datasets


In [1]:
import os
from datasets import load_dataset, Dataset
import random

# Set full Hugging Face cache directory override
custom_hf_home = os.path.abspath("../data/content/hf_home")
os.environ["HF_HOME"] = os.environ.get("HF_HOME")
os.environ["HF_DATASETS_CACHE"] = os.environ.get("HF_DATASETS_CACHE")

langs = ["python", "php", "javascript", "go", "ruby", "java"]
output_dir = "../data/content/datasets/"
os.makedirs(output_dir, exist_ok=True)

all_samples = []
total_samples = 0

for lang in langs:
    print(f"\n🔽 Loading the-stack-smol for: {lang}")
    try:
        ds = load_dataset("codeparrot/codeparrot-clean-train", split="train[:5%]", cache_dir=os.environ["HF_DATASETS_CACHE"])
        print(f"✅ {lang}: {len(ds)} samples")
    except Exception as e:
        print(f"❌ Failed to load {lang}: {e}")
        continue

    ds = ds.shuffle(seed=42).select(range(min(10000, len(ds))))
    print(f"🔢 Now using {len(ds)} samples for {lang}")

    for s in ds:
        s["language"] = lang
        all_samples.append(s)
    total_samples += len(ds)

output_path = os.path.join(output_dir, "stack_smol_combined.jsonl")
Dataset.from_list(all_samples).to_json(output_path)
print(f"📦 Saved {total_samples} samples to {output_path}")


/home/xon/miniconda3/envs/qlora-py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🔽 Loading the-stack-smol for: python


Repo card metadata block was not found. Setting CardData to empty.


✅ python: 265000 samples
🔢 Now using 10000 samples for python

🔽 Loading the-stack-smol for: php


Repo card metadata block was not found. Setting CardData to empty.


...

✅ ruby: 265000 samples
🔢 Now using 10000 samples for ruby

🔽 Loading the-stack-smol for: java


Repo card metadata block was not found. Setting CardData to empty.


✅ java: 265000 samples
🔢 Now using 10000 samples for java


Creating json from Arrow format: 100%|██████████| 60/60 [00:05<00:00, 11.07ba/s]

📦 Saved 60000 samples to ../data/content/datasets/stack_smol_combined.jsonl


In [9]:
output_path = os.path.join(output_dir, "stack_smol_combined.jsonl")
Dataset.from_list(all_samples).to_json(output_path)
print(f"📦 Saved {total_samples} samples to {output_path}")

Creating json from Arrow format: 100%|██████████| 60/60 [00:05<00:00, 11.57ba/s]

📦 Saved 60000 samples to ../data/content/datasets/stack_smol_combined.jsonl


In [10]:
import pandas as pd 

In [11]:
oData =pd.read_json(output_path, lines=True)

In [12]:
oData.head(5)

,repo_name,path,copies,size,content,license,hash,line_mean,line_max,alpha_frac,autogenerated,language
0,irskep/computerwords,tests/kissup/test_lexer.py,1,3741,import unittest\nfrom textwrap import dedent\n...,bsd-3-clause,-5183642910038422601,37.567010,100,0.526597,False,python
1,awest1339/multiscanner,utils/api.py,1,30509,#!/usr/bin/env python\n'''\nTHIS APP IS NOT PR...,mpl-2.0,3825134890661274895,32.198041,122,0.607952,False,python
2,RAtechntukan/CouchPotatoServer,couchpotato/core/media/movie/providers/trailer...,9,1609,# -*- coding: utf-8 -*-\nfrom __future__ impor...,gpl-3.0,2890011466795254799,32.869565,115,0.525032,False,python
3,saurabh6790/aimobilize-app-backup,setup/doctype/email_settings/email_settings.py,29,2140,"# Copyright (c) 2013, Web Notes Technologies P...",agpl-3.0,-6215540554645841904,30.940299,93,0.716355,False,python
4,shoopio/shoop,shuup_tests/discounts/test_catalog_campaign_im...,2,8268,# -*- coding: utf-8 -*-\n# This file is part o...,agpl-3.0,5962903158220169458,44.180328,121,0.746976,False,python


In [28]:
import json

# Check if required columns exist
if 'language' in oData.columns and 'content' in oData.columns:
    # Filter Python code entries
    python_code_df = oData[oData['language'] == 'python']

    # Format instruction-response pairs
    formatted_data = []
    for _, row in python_code_df.iterrows():
        code = row['content']
        path = row['path'] if 'path' in row and isinstance(row['path'], str) else "unknown_path.py"
        filename = path.split("/")[-1]  # Just the filename (e.g., config.php)

        text_entry = {
            "text": f"### Instruction:\nExplain what this file `{filename}` does.\n\n### Response:\n{code.strip()}"
        }
        formatted_data.append(text_entry)

    # Save as JSONL
    output_jsonl_path = "../data/content/datasets/extracted_python_code.jsonl"
    with open(output_jsonl_path, "w", encoding="utf-8") as f:
        for item in formatted_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Saved {len(formatted_data)} instruction-response entries to {output_jsonl_path}")
else:
    print("❌ The required columns ('language' and 'content') are not present in the dataset.")


✅ Saved 10000 instruction-response entries to ../data/content/datasets/extracted_python_code.jsonl


In [15]:
print(oData['content'][0])

import unittest
from textwrap import dedent

from computerwords.markdown_parser import CFMParserConfig
from computerwords.markdown_parser import html_lexer
from computerwords.markdown_parser import tokens as t
from computerwords.markdown_parser import ast
from computerwords.markdown_parser import parser_support
from computerwords.markdown_parser.src_loc import (
    SourceLocation as L,
    SourceRange as R,
)


parse_funcs = parser_support.PARSE_FUNC_REGISTRY


def lex(s):
    config = CFMParserConfig(
        document_id=('test.md',), document_path='test.md', allowed_tags=set())
    return list(html_lexer.lex_html(config, s))


def strip(s):
    return dedent(s)[1:-1]


class TestLexer(unittest.TestCase):
    def test_simple(self):
        self.assertEqual(lex('<'), [
            t.BracketLeftToken( L(0, 0, 0).plus(1)),
            t.EndToken(         L(0, 1, 1).plus(0)),
        ])
        self.assertEqual(lex('a'), [
            t.TextToken(        L(0, 0, 0).plus(1), 'a'),
       